In [2]:
%reload_ext autoreload
%autoreload 2

import warnings, os, sys
warnings.filterwarnings('ignore')
from IPython.display import clear_output

import numpy as np
import pandas as pd
import swifter
from functools import reduce
from tqdm.auto import tqdm

from rdkit import Chem
from rdkit.Chem import PandasTools
PandasTools.RenderImagesInAllDataFrames(images=True)
from rdkit import RDLogger; RDLogger.DisableLog('rdApp.*')
from rdkit.DataStructs import TanimotoSimilarity
from rdkit.Chem.AllChem import GetMorganFingerprintAsBitVect
from rdkit.Chem import AllChem
from rdkit.Chem import Descriptors

from rdkit.Chem.Draw import IPythonConsole
IPythonConsole.ipython_useSVG = True
IPythonConsole.molSize = 600, 400
IPythonConsole.drawOptions.addAtomIndices = False

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook"
pio.renderers.default_config = {
    "toImageButtonOptions": {"format": "png", "scale": 5}}

# sys.path.append(os.path.abspath(".."))
# from new_geass import Geass
# from AIDDTools import fingerprint_tools as fpt

clear_output()

In [3]:
from rdkit.Chem import rdFMCS, rdMolDescriptors
from rdkit.Chem.MolStandardize import rdMolStandardize

# Create normaliser and uncharger
normalizer = rdMolStandardize.Normalizer()
uncharger = rdMolStandardize.Uncharger()

def standardize_mol(mol):
    if mol is None:
        return None
    mol = normalizer.normalize(mol)   # normalise functional groups
    mol = uncharger.uncharge(mol)     # neutralise charges
    mol = rdMolStandardize.Cleanup(mol)
    return mol

In [ ]:
from rdkit.Chem import rdMolDescriptors
sys.path.append(os.path.abspath(R"C:\\Users\\Idener\\Sync\\CODE"))
from AIDDTools import fingerprint_tools as fpt

def calc_feature_vec(mol, fp='atompair'):
    fp_vec = fpt.calculate_fp(mol, method=fp, as_numpy=True)

    desc_vec = np.array([
        rdMolDescriptors.CalcNumRings(mol),
        rdMolDescriptors.CalcNumAromaticRings(mol),
        rdMolDescriptors.CalcNumAliphaticRings(mol),
        sum(1 for a in mol.GetAtoms() if a.GetAtomicNum() == 6 and a.GetIsAromatic()),
        sum(1 for a in mol.GetAtoms() if a.GetAtomicNum() == 6 and not a.GetIsAromatic())])

    return np.concatenate([desc_vec, fp_vec])

In [20]:
# Compile the Building Unit Library (BUL)
df_kegg = pd.read_parquet('data/Database_KEGG_cpd.gzip', engine='fastparquet')
df_kegg = df_kegg[~df_kegg['ISOSMILES'].str.contains('P|\\*', na=False)]

df_kegg = df_kegg.dropna(subset='ISOSMILES')
PandasTools.AddMoleculeColumnToFrame(df_kegg, smilesCol='ISOSMILES')
df_kegg = df_kegg.dropna(subset='ROMol')
df_kegg['ROMol'] = df_kegg['ROMol'].swifter.apply(standardize_mol)
df_kegg['ISOSMILES'] = df_kegg['ROMol'].swifter.apply(Chem.MolToSmiles)
df_kegg = df_kegg.drop_duplicates(subset='ISOSMILES').reset_index(drop=True)

df_kegg['InChI'] = df_kegg['ROMol'].swifter.apply(Chem.MolToInchi)
df_kegg = df_kegg.dropna(subset=['ISOSMILES', 'InChI']).reset_index(drop=True)

df_kegg['Formula'] = df_kegg['ROMol'].swifter.apply(AllChem.CalcMolFormula)

df_kegg = df_kegg.rename(columns={'ENTRY':'ID', 'MOL_WEIGHT':'MW'})
df_kegg = df_kegg[['ID', 'ISOSMILES', 'Formula', 'MW', 'InChI', 'BRITE', 'ROMol']]
df_kegg = df_kegg.rename(columns={'BRITE':'Ontology'})
clear_output()

In [ ]:
df_LM = pd.read_pickle('data/Database_lipidmaps.pkl')
df_LM = df_LM.rename(columns={'FORMULA':'Formula', 'ONTOLOGY':'Ontology'})
df_LM['InChI'] = df_LM['ROMol'].swifter.apply(Chem.MolToInchi)
df_LM['Ontology'] = df_LM['Ontology'].swifter.apply(lambda x: '; '.join(x) if isinstance(x, list) else x)

df_BUL = pd.concat([df_kegg, df_LM], ignore_index=True).drop_duplicates(subset='ISOSMILES').reset_index(drop=True)
df_BUL = df_BUL.dropna(subset=['MW', 'InChI']).reset_index(drop=True)
df_BUL = df_BUL[~df_BUL['ISOSMILES'].str.contains(R'\.')]
df_BUL['MW'] = df_BUL['MW'].round(3)
df_BUL['Num_Atoms'] = df_BUL['ROMol'].swifter.apply(lambda x: x.GetNumHeavyAtoms())
df_BUL['Feature_vec'] = df_BUL['ROMol'].swifter.apply(calc_feature_vec)

df_BUL = df_BUL[['ID', 'ISOSMILES', 'Formula', 'MW',
    'Num_Atoms', 'InChI', 'Ontology', 'Feature_vec']]
df_BUL.drop(columns=['Feature_vec']).to_csv('data/Database_BUL.csv', index=False, sep='\t')
df_BUL.to_pickle('data/Database_BUL.pkl')
clear_output()

df_BUL

,ID,ISOSMILES,Formula,MW,Num_Atoms,InChI,Ontology,Feature_vec
0,C00001,O,H2O,18.015,1,InChI=1S/H2O/h1H2,Therapeutic category of drugs in Japan [BR:br0...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,C00007,[O],O,31.999,1,InChI=1S/O,Anatomical Therapeutic Chemical (ATC) classifi...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,C00011,O=C=O,CO2,44.010,3,InChI=1S/CO2/c2-1-3,Anatomical Therapeutic Chemical (ATC) classifi...,"[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,C00014,N,H3N,17.030,1,InChI=1S/H3N/h1H3,Risk category of Japanese OTC drugs [BR:br0831...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,C00019,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,C15H23N6O5S+,398.437,27,InChI=1S/C15H22N6O5S/c1-27(3-2-7(16)15(24)25)4...,Compounds with biological roles [BR:br08001]\n...,"[3, 2, 1, 5, 10, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0,..."
...,...,...,...,...,...,...,...,...
60707,LMPK15050002,CCCCCCCCCCC[C@H](CC1=C(O)C(=O)C=C(OC)C1=O)OC(C)=O,C22H34O6,394.508,28,InChI=1S/C22H34O6/c1-4-5-6-7-8-9-10-11-12-13-1...,Phenolic lipids [PK15]; Alkyl hydroquinones an...,"[1, 0, 1, 0, 22, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0,..."
60708,LMPK15050003,CCCCCCCCCCCCC[C@H](CC1=C(O)C(=O)C=C(OC)C1=O)OC...,C24H38O6,422.562,30,InChI=1S/C24H38O6/c1-4-5-6-7-8-9-10-11-12-13-1...,Phenolic lipids [PK15]; Alkyl hydroquinones an...,"[1, 0, 1, 0, 24, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0,..."
60709,LMPK15050004,CCCCCCCCCCCCC[C@H](CC1=CC(=O)C=C(OC)C1=O)OC(C)=O,C24H38O5,406.563,29,InChI=1S/C24H38O5/c1-4-5-6-7-8-9-10-11-12-13-1...,Phenolic lipids [PK15]; Alkyl hydroquinones an...,"[1, 0, 1, 0, 24, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0,..."
60710,LMPK15050005,CCCCCCCCCCC[C@H](CC1=CC(=O)C=C(OC)C1=O)OC(C)=O,C22H34O5,378.509,27,InChI=1S/C22H34O5/c1-4-5-6-7-8-9-10-11-12-13-2...,Phenolic lipids [PK15]; Alkyl hydroquinones an...,"[1, 0, 1, 0, 22, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0,..."
